In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, classification_report, accuracy_score

# 1. MEMUAT DATA DAN MANAJEMEN DATA (DATA MANAGEMENT)
df = pd.read_csv('Amazon Sale Report.csv', low_memory=False)

# Mengisi missing values pada Amount dengan median per kategori
df['Amount'] = df.groupby('Category')['Amount'].transform(lambda x: x.fillna(x.median()))
# Menyederhanakan status menjadi Sukses (Shipped/Delivered) vs Batal (Cancelled)
df = df[df['Status'].isin(['Shipped - Delivered to Buyer', 'Cancelled', 'Shipped'])].copy()
df['Is_Cancelled'] = df['Status'].apply(lambda x: 1 if x == 'Cancelled' else 0)

# Feature Engineering: Mengambil data numerik dasar
df_clean = df[['Qty', 'Amount', 'Category', 'Fulfilment', 'ship-state', 'Is_Cancelled']].dropna()

print("--- MANAJEMEN DATA SELESAI ---")
print(f"Ukuran data siap pakai untuk Machine Learning: {df_clean.shape}\n")


# 2. ASOSIASI DAN KORELASI DATA
# Menghitung korelasi Spearman antara Qty dan Amount karena data keuangan bersifat miring (skewed)
korelasi = df_clean[['Qty', 'Amount']].corr(method='spearman')
print("--- MATRIKS KORELASI (SPEARMAN) ---")
print(korelasi, "\n")


# 3. ANALISIS REGRESI: MEMPREDIKSI AMOUNT BERDASARKAN QTY
X_reg = df_clean[['Qty']]
y_reg = df_clean['Amount']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

model_reg = LinearRegression()
model_reg.fit(X_train_r, y_train_r)
y_pred_r = model_reg.predict(X_test_r)

print("--- HASIL ANALISIS REGRESI ---")
print(f"Koefisien Regresi (Slope): {model_reg.coef_[0]:.2f}")
print(f"R-squared Score: {r2_score(y_test_r, y_pred_r):.4f}\n")


# 4. KLASIFIKASI DATA: MEMPREDIKSI APAKAH PESANAN AKAN BATAL (IS_CANCELLED)
# Melakukan encoding untuk data teks (kategorikal) menjadi angka
le = LabelEncoder()
df_clean['Category_Encoded'] = le.fit_transform(df_clean['Category'])
df_clean['Fulfilment_Encoded'] = le.fit_transform(df_clean['Fulfilment'])

X_class = df_clean[['Qty', 'Amount', 'Category_Encoded', 'Fulfilment_Encoded']]
y_class = df_clean['Is_Cancelled']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

model_class = LogisticRegression(max_iter=1000)
model_class.fit(X_train_c, y_train_c)
y_pred_c = model_class.predict(X_test_c)

print("--- HASIL EVALUASI MODEL KLASIFIKASI (PREDIKSI BATAL) ---")
print(f"Akurasi Model: {accuracy_score(y_test_c, y_pred_c)*100:.2f}%")
print(classification_report(y_test_c, y_pred_c))


# 5. CLUSTERING DATA: SEGMENTASI GEOGRAFIS NEGARA BAGIAN
# Mengagregasikan data total revenue dan rata-rata Qty per negara bagian (ship-state)
state_market = df_clean.groupby('ship-state').agg({'Amount':'sum', 'Qty':'mean'}).reset_index()

# Standarisasi fitur agar skala seimbang
scaler = StandardScaler()
scaled_features = scaler.fit_transform(state_market[['Amount', 'Qty']])

# Melakukan K-Means dengan 3 Cluster (High, Medium, Low Performers)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) # Added n_init to avoid warning
state_market['Cluster'] = kmeans.fit_predict(scaled_features) # Changed from fit_transform to fit_predict

print("--- HASIL SEGMENTASI PASAR (CLUSTERING) ---")
print(state_market.groupby('Cluster').agg({'ship-state':'count', 'Amount':'mean', 'Qty':'mean'}))

--- MANAJEMEN DATA SELESAI ---
Ukuran data siap pakai untuk Machine Learning: (124875, 6)

--- MATRIKS KORELASI (SPEARMAN) ---
             Qty    Amount
Qty     1.000000  0.017902
Amount  0.017902  1.000000 

--- HASIL ANALISIS REGRESI ---
Koefisien Regresi (Slope): 46.32
R-squared Score: 0.0049

--- HASIL EVALUASI MODEL KLASIFIKASI (PREDIKSI BATAL) ---
Akurasi Model: 95.21%
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     21234
           1       0.99      0.69      0.81      3741

    accuracy                           0.95     24975
   macro avg       0.97      0.84      0.89     24975
weighted avg       0.95      0.95      0.95     24975

--- HASIL SEGMENTASI PASAR (CLUSTERING) ---
         ship-state        Amount       Qty
Cluster                                    
0                61  5.805997e+05  0.916784
1                 5  9.060183e+06  0.904145
2                 2  1.072455e+03  0.000000
